In [28]:
from pathlib import Path
import re
import numpy as np

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer, CrossEncoder

import faiss
from rank_bm25 import BM25Okapi

import ollama

print("Libraries Loaded")

Libraries Loaded


In [29]:
data_path = Path(r"C:\Users\USER\kalashala_AI\data")

md_files = list(data_path.glob("*.md"))

print(f"Found {len(md_files)} markdown files\n")

for file in md_files:
    print(file.name)

Found 6 markdown files

academy.md
contacts.md
courses.md
faq.md
policies.md
trainers.md


In [30]:
documents = []

for file in md_files:

    with open(file, "r", encoding="utf-8") as f:

        text = f.read()

    documents.append({

        "file": file.name,

        "category": file.stem,

        "text": text

    })

print("Documents Loaded:", len(documents))

Documents Loaded: 6


In [31]:
for doc in documents:

    print("=" * 80)

    print(doc["file"])

    print()

    print(doc["text"][:500])

    print()

academy.md

# Academy Name:
Kalashala Makeup Academy

## Location
Address:
Hyderabad, Near Madhapur, Hi-Tech City
## Working Hours
9 AM – 7 PM
## Contact
Phone:
9876543210

Email:
info@kalashala.com

## Founder
krishna

## Parking
Available

## Hostel
Not Available

## Language
English, Telugu, Hindi

contacts.md

# Contact Information

For admissions, course registration, seat reservation,
appointment booking and counselling please contact us.

Phone:
+91-9876543210

Email:
info@kalashala.com

Website:
https://kalashala.com

Instagram:
https://instagram.com/kalashala

Facebook:
https://facebook.com/kalashala

courses.md

# Professional Makeup Course

Duration:
45 Days

Fees:
₹35,000

Mode:
Offline

Eligibility:
No prior experience required.

Topics Covered:
- Skin Preparation
- Foundation Matching
- Bridal Makeup
- Party Makeup
- HD Makeup
- Airbrush Makeup
- Saree Draping
- Hairstyling
- Client Consultation

Certification:
Certificate provided after successful completion.

Batch Size

In [32]:
headers_to_split_on = [

    ("#", "H1"),

    ("##", "H2"),

    ("###", "H3")

]

markdown_splitter = MarkdownHeaderTextSplitter(

    headers_to_split_on=headers_to_split_on,

    strip_headers=False

)

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=100
)

all_chunks = []

chunk_id = 0

for doc in documents:

    md_chunks = markdown_splitter.split_text(doc["text"])

    for md_chunk in md_chunks:

        header = ""

        if "H3" in md_chunk.metadata:

            header = md_chunk.metadata["H3"]

        elif "H2" in md_chunk.metadata:

            header = md_chunk.metadata["H2"]

        elif "H1" in md_chunk.metadata:

            header = md_chunk.metadata["H1"]

        pieces = text_splitter.split_text(md_chunk.page_content)

        for piece in pieces:

            all_chunks.append({

                "chunk_id": chunk_id,

                "file": doc["file"],

                "category": doc["category"],

                "header": header,

                "text": piece

            })

            chunk_id += 1

print("Total Chunks:", len(all_chunks))

Total Chunks: 19


In [33]:
for chunk in all_chunks:

    print("="*100)

    print("Chunk ID :", chunk["chunk_id"])

    print("File     :", chunk["file"])

    print("Category :", chunk["category"])

    print("Header   :", chunk["header"])

    print()

    print(chunk["text"][:500])

    print()

    if chunk["chunk_id"] == 10:
        break

Chunk ID : 0
File     : academy.md
Category : academy
Header   : Academy Name:

# Academy Name:
Kalashala Makeup Academy

Chunk ID : 1
File     : academy.md
Category : academy
Header   : Location

## Location
Address:
Hyderabad, Near Madhapur, Hi-Tech City

Chunk ID : 2
File     : academy.md
Category : academy
Header   : Working Hours

## Working Hours
9 AM – 7 PM

Chunk ID : 3
File     : academy.md
Category : academy
Header   : Contact

## Contact
Phone:
9876543210  
Email:
info@kalashala.com

Chunk ID : 4
File     : academy.md
Category : academy
Header   : Founder

## Founder
krishna

Chunk ID : 5
File     : academy.md
Category : academy
Header   : Parking

## Parking
Available

Chunk ID : 6
File     : academy.md
Category : academy
Header   : Hostel

## Hostel
Not Available

Chunk ID : 7
File     : academy.md
Category : academy
Header   : Language

## Language
English, Telugu, Hindi

Chunk ID : 8
File     : contacts.md
Category : contacts
Header   : Contact Information

# Contact Inf

In [34]:
embedding_model = SentenceTransformer(

    "BAAI/bge-base-en-v1.5"

)

print("Embedding Model Loaded!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding Model Loaded!


In [35]:
texts = [

    chunk["text"]

    for chunk in all_chunks

]

embeddings = embedding_model.encode(

    texts,

    normalize_embeddings=True,

    show_progress_bar=True

)

embeddings = np.array(

    embeddings,

    dtype="float32"

)

print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Shape: (19, 768)


In [36]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

Vectors Stored: 19


In [37]:
import re

def tokenize(text):

    text = text.lower()

    return re.findall(r"[a-zA-Z0-9]+", text)

In [38]:
print(tokenize("Professional Makeup Course (₹35,000)"))

['professional', 'makeup', 'course', '35', '000']


In [39]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [

    tokenize(chunk["text"])

    for chunk in all_chunks

]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 Ready!")

BM25 Ready!


In [40]:
def semantic_search(query, k=10):

    query_embedding = embedding_model.encode(

        [query],

        normalize_embeddings=True

    )

    query_embedding = np.array(

        query_embedding,

        dtype="float32"

    )

    scores, indices = index.search(

        query_embedding,

        k

    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({

            "chunk": all_chunks[idx],

            "score": float(score)

        })

    return results

In [41]:
def bm25_search(query, k=10):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for idx in top_indices:

        results.append({

            "chunk": all_chunks[idx],

            "score": float(scores[idx])

        })

    return results

In [42]:
results = semantic_search("Who is the fee structure for courses?")

for r in results:

    print("="*80)

    print("Score :", round(r["score"],4))

    print("File :", r["chunk"]["file"])

    print("Header :", r["chunk"]["header"])

    print()

    print(r["chunk"]["text"][:400])

Score : 0.6641
File : policies.md
Header : 

# Admission Policy  
##
Enrollment is confirmed only after payment.  
------------------------------## --  
Attendance Policy  
Minimum attendance:
80%  
-----------------------## ---------  
Refund Policy  
Registration fee is non-refundable.  
-----------------## ---------------  
Course Transfer  
Students may transfer to another batch with prior approval.  
-----------## --------------------- 
Score : 0.638
File : courses.md
Header : Hair Styling Course

# Hair Styling Course  
Duration:
30 Days  
Fees:
₹18,000  
Topics:
- Hair Spa
- Hair Cutting
- Hair Coloring
- Hair Styling
Score : 0.5948
File : faq.md
Header : Will I get a certificate?

### Will I get a certificate?  
Yes.  
--------------------------------
Score : 0.5921
File : faq.md
Header : Can I pay in EMI?

### Can I pay in EMI?  
Yes.  
--------------------------------
Score : 0.5917
File : faq.md
Header : Is accommodation available?

### Is accommodation available?  
No.  
--

In [43]:
results = bm25_search("Who is the fee structure for courses?")

for r in results:

    print("="*80)

    print("Score :", round(r["score"],4))

    print("File :", r["chunk"]["file"])

    print("Header :", r["chunk"]["header"])

    print()

    print(r["chunk"]["text"][:400])

Score : 3.9085
File : policies.md
Header : 

# Admission Policy  
##
Enrollment is confirmed only after payment.  
------------------------------## --  
Attendance Policy  
Minimum attendance:
80%  
-----------------------## ---------  
Refund Policy  
Registration fee is non-refundable.  
-----------------## ---------------  
Course Transfer  
Students may transfer to another batch with prior approval.  
-----------## --------------------- 
Score : 2.8797
File : faq.md
Header : Is accommodation available?

### Is accommodation available?  
No.  
--------------------------------
Score : 2.0987
File : faq.md
Header : What should I bring on the first day?

### What should I bring on the first day?  
Notebook and ID proof.
Score : 1.494
File : contacts.md
Header : Contact Information

# Contact Information  
For admissions, course registration, seat reservation,
appointment booking and counselling please contact us.  
Phone:
+91-9876543210  
Email:
info@kalashala.com  
Website:
https://ka

In [44]:
SYNONYMS = {

    "cost":"fees",

    "price":"fees",

    "started":"founder",

    "owner":"founder",

    "teacher":"trainer",

    "teach":"trainer",

    "duration":"course duration",

    "certificate":"certification",

    "join":"enroll",

    "timings":"working hours"

}

In [45]:
def hybrid_search(query, k=10):

    # Get more candidates for better fusion
    semantic_results = semantic_search(query, k=20)
    bm25_results = bm25_search(query, k=20)

    rrf_scores = {}
    chunk_lookup = {}

    # Reciprocal Rank Fusion constant
    K = 60

    # Semantic rankings
    for rank, result in enumerate(semantic_results):

        chunk = result["chunk"]
        chunk_id = chunk["chunk_id"]

        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (K + rank)

        chunk_lookup[chunk_id] = chunk

    # BM25 rankings
    for rank, result in enumerate(bm25_results):

        chunk = result["chunk"]
        chunk_id = chunk["chunk_id"]

        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (K + rank)

        chunk_lookup[chunk_id] = chunk

    # Sort by fused score
    ranked = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for chunk_id, score in ranked[:k]:

        results.append({

            "chunk": chunk_lookup[chunk_id],

            "score": score

        })

    return results

In [46]:
query = "What is the fee for professional makeup course?"

results = hybrid_search(query)

for result in results:

    chunk = result["chunk"]

    print("="*100)

    print("Hybrid Score :", round(result["score"],4))

    print("File :", chunk["file"])

    print("Header :", chunk["header"])

    print()

    print(chunk["text"][:500])

Hybrid Score : 0.0437
File : trainers.md
Header : 

# Trainer  
##
Priya  
Experience:
8 Years  
Specialization:
Bridal Makeup  
--------------------------------  
Trainer:
Anjali  
Experience:
5 Years  
Specialization:
Hair Styling
Hybrid Score : 0.0328
File : courses.md
Header : Professional Makeup Course

# Professional Makeup Course  
Duration:
45 Days  
Fees:
₹35,000  
Mode:
Offline  
Eligibility:
No prior experience required.  
Topics Covered:
- Skin Preparation
- Foundation Matching
- Bridal Makeup
- Party Makeup
- HD Makeup
- Airbrush Makeup
- Saree Draping
- Hairstyling
- Client Consultation  
Certification:
Certificate provided after successful completion.  
Batch Size:
15 Students  
Timings:
10 AM – 1 PM  
Next Batch:
5 August 2026
Hybrid Score : 0.0325
File : policies.md
Header : 

# Admission Policy  
##
Enrollment is confirmed only after payment.  
------------------------------## --  
Attendance Policy  
Minimum attendance:
80%  
-----------------------## ---------  
Ref

In [47]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(

    "BAAI/bge-reranker-base"

)

print("Reranker Loaded!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reranker Loaded!


In [48]:
def rerank(query, retrieved_docs):

    pairs = [

        (query, doc["chunk"]["text"])

        for doc in retrieved_docs

    ]

    scores = reranker.predict(pairs)

    ranked = sorted(

        zip(scores, retrieved_docs),

        key=lambda x: x[0],

        reverse=True

    )

    return ranked

In [49]:
query = "Where is the address?"

retrieved_docs = hybrid_search(query, k=10)

ranked_docs = rerank(query, retrieved_docs)

for score, doc in ranked_docs:

    print("=" * 100)

    print("Rerank Score :", round(float(score), 4))

    print("File :", doc["chunk"]["file"])

    print("Header :", doc["chunk"]["header"])

    print()

    print(doc["chunk"]["text"][:500])

Rerank Score : 0.3599
File : academy.md
Header : Location

## Location
Address:
Hyderabad, Near Madhapur, Hi-Tech City
Rerank Score : 0.0004
File : contacts.md
Header : Contact Information

# Contact Information  
For admissions, course registration, seat reservation,
appointment booking and counselling please contact us.  
Phone:
+91-9876543210  
Email:
info@kalashala.com  
Website:
https://kalashala.com  
Instagram:
https://instagram.com/kalashala  
Facebook:
https://facebook.com/kalashala
Rerank Score : 0.0001
File : academy.md
Header : Parking

## Parking
Available
Rerank Score : 0.0001
File : faq.md
Header : Is accommodation available?

### Is accommodation available?  
No.  
--------------------------------
Rerank Score : 0.0
File : faq.md
Header : What should I bring on the first day?

### What should I bring on the first day?  
Notebook and ID proof.
Rerank Score : 0.0
File : trainers.md
Header : 

# Trainer  
##
Priya  
Experience:
8 Years  
Specialization:
Bridal Makeup  
---

In [50]:
evaluation_data = []

while True:

    question = input("\nAsk: ")

    if question.lower() == "exit":
        break

    # -----------------------------
    # Retrieve Documents
    # -----------------------------

    retrieved_docs = hybrid_search(question, k=15)

    ranked_docs = rerank(question, retrieved_docs)

    # -----------------------------
    # Build Context
    # -----------------------------

    context = ""

    for score, doc in ranked_docs[:5]:

        context += f"""
Document : {doc['chunk']['file']}
Section  : {doc['chunk']['header']}

{doc['chunk']['text']}

{'='*80}

"""
    prompt = f"""
You are the AI assistant for Kalashala Academy.

Use ONLY the provided context.

If the user asks about:

• courses
• fees
• trainers
• timings
• duration
• certification
• locations
• batches

answer normally.

If the user wants to:

• join a course
• register
• enroll
• reserve a seat
• book an appointment
• admission

then politely provide the contact information available in the context.

Never invent phone numbers or links.

If information is unavailable,
say you don't know.

Context:

{context}

Question:

{question}

Answer:
"""
    response = ollama.chat(

        model="llama3",

        messages=[

            {

                "role":"user",

                "content":prompt

            }

        ],

        options={

            "temperature":0.2

        }

    )

    answer = response["message"]["content"]

    print("\n")
    print("="*100)
    print("ANSWER")
    print("="*100)

    print(answer)
    evaluation_data.append({

        "question": question,

        "answer": answer,

        "context": context

    })


Ask:  i wnat to enroll the course




ANSWER
I'd be happy to help! However, I need to inform you that we don't have a direct enrollment process through me. For admissions, course registration, seat reservation, appointment booking and counselling please contact us.

You can reach out to us at:

Phone: +91-9876543210
Email: info@kalashala.com
Website: https://kalashala.com
Instagram: https://instagram.com/kalashala
Facebook: https://facebook.com/kalashala

They will be able to guide you through the enrollment process and answer any questions you may have.



Ask:  thank you




ANSWER
You're welcome! It was my pleasure to assist you. If you have any further questions or need assistance with anything else, feel free to ask!



Ask:  exit
